In [ ]:
!pip install openai matplotlib koreanize-matplotlib

In [ ]:
# ============================================================
# 실습 2-5. RAG 브리지 - API 한계 체험
# 목표: 내부 문서 부재·환각을 체험하고 문서 주입으로 해결 원리를 이해한다
# 사전 설치: pip install openai matplotlib koreanize-matplotlib
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False

# ── API 키 설정 ──────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Colab Secrets에서 API 키 로드 완료")
except Exception:
    pass
# os.environ["OPENAI_API_KEY"] = "sk-..."

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "API 키가 설정되지 않았습니다.\n"
        "방법 A: 왼쪽 사이드바 열쇠 아이콘 -> 'OPENAI_API_KEY' 등록\n"
        "방법 B: os.environ['OPENAI_API_KEY'] = 'sk-...' 주석 해제"
    )

from openai import OpenAI
client = OpenAI()
print(f"API 키 확인: sk-...{os.environ['OPENAI_API_KEY'][-4:]}")


# ============================================================
# [실습 변수] 여기만 수정하여 다양한 실험 가능
# ============================================================

# 한계 1: 기관 내부 정보 질의 (본인 기관 관련 질문으로 교체 가능)
INTERNAL_QUESTION = "우리 기관의 2024년 연구비 집행 지침 개정 내용은 무엇인가요?"

# 한계 2: 미발표 연구 데이터 환각 유발 질문
# 존재하지 않는 논문/데이터에 대한 질문 -> 환각 발생 가능성 확인
HALLUC_QUESTION = (
    "김연구 박사가 2024년 발표한 '한국어 멀티레이블 분류 벤치마크' "
    "데이터셋의 규모와 구성은?"
)

# RAG 해결 방식: 문서 컨텍스트 직접 주입
# 실제 기관 내부 문서 내용으로 교체 가능
REAL_CONTEXT = """
[2024년 하반기 연구비 집행 지침 개정 사항]
- 개정일: 2024년 7월 1일
- 주요 변경: 간접비율 상한 25% -> 30% 조정
- 신규 항목: AI 소프트웨어 라이선스 비용 직접비 인정
- 집행 기한: 과제 종료 후 30일 이내 (기존 15일)
"""

# RAG 답변 온도: 사실 확인 목적 -> 낮게 유지
RAG_TEMPERATURE = 0.1

# 환각 실험 추가: 높은 온도에서 환각 발생 여부 비교
# True로 설정 시 동일 질문을 높은 온도로 재실행
RUN_HALLUC_TEMP_EXPERIMENT = True
HALLUC_HIGH_TEMP = 1.2   # 높은 온도에서 환각 발생 가능성 증가

# API vs RAG 적합도 점수 (1~5)
# 본인 평가로 수정 가능
API_SCORES = {
    "최신 정보": 2, "내부 문서": 1, "근거 제시": 2,
    "즉시 사용": 5, "커스터마이징": 3,
}
RAG_SCORES = {
    "최신 정보": 4, "내부 문서": 5, "근거 제시": 5,
    "즉시 사용": 4, "커스터마이징": 5,
}

# ============================================================
# 이하 코드는 수정 불필요
# ============================================================

# ── 공통 함수 ─────────────────────────────────────────────────
def chat(system_prompt, user_prompt,
         temperature=RAG_TEMPERATURE, max_tokens=400):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content, resp.usage

def section(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")


# ── 블록 1: 한계 1 - 기관 내부 정보 부재 ──────────────────────
section("블록 1. 한계 1 - 기관 내부 정보 부재")

# 엄격 프롬프트: 모르면 모른다고 답하도록 지시
# 없으면 그럴듯하게 지어내는 환각 방지 목적
sys_strict = "정확한 정보만 답하세요. 모르면 '모릅니다'라고 하세요."

ans_internal, _ = chat(
    system_prompt=sys_strict,
    user_prompt=INTERNAL_QUESTION,
    temperature=RAG_TEMPERATURE,
)
print(f"\n질문: {INTERNAL_QUESTION}")
print(f"\nAPI 응답:\n{ans_internal}")
print("\n-> 내부 문서 미학습 - 답변 불가 또는 일반적 내용만 반환")
print("   실제 중요한 내부 정보를 물었을 때 API 단독으로는 신뢰하기 어렵습니다.")


# ── 블록 2: 한계 2 - 미발표 데이터 환각 체험 ──────────────────
section("블록 2. 한계 2 - 미발표 연구 데이터 환각")

# 기본 실험: 일반 온도
ans_halluc, _ = chat(
    system_prompt="",
    user_prompt=HALLUC_QUESTION,
    temperature=0.7,
)
print(f"\n질문: {HALLUC_QUESTION}")
print(f"\nAPI 응답 (T=0.7):\n{ans_halluc}")

# 추가 실험: 높은 온도에서 환각 발생 여부 비교
if RUN_HALLUC_TEMP_EXPERIMENT:
    print(f"\n--- 높은 온도 실험 (T={HALLUC_HIGH_TEMP}) ---")
    print("온도가 높을수록 낮은 확률 토큰도 선택 -> 환각 발생 가능성 증가")
    ans_halluc_high, _ = chat(
        system_prompt="",
        user_prompt=HALLUC_QUESTION,
        temperature=HALLUC_HIGH_TEMP,
    )
    print(f"\nAPI 응답 (T={HALLUC_HIGH_TEMP}):\n{ans_halluc_high}")
    print("\n-> T=0.7 vs T=1.2 비교: 어느 쪽에서 더 구체적인 내용이 생성되는가?")
    print("   구체적일수록 환각 가능성 높음 (실제로 존재하지 않는 논문)")

print("\n-> 환각 위험: 모델이 알지 못하는 내용을 그럴듯하게 생성")
print("   특히 연구 결과·수치·인명 등 검증 가능한 사실에서 위험")


# ── 블록 3: RAG 해결 방식 - 문서 컨텍스트 주입 ────────────────
section("블록 3. RAG 해결 방식 - 문서 컨텍스트 직접 주입")

# 핵심 원리:
# 1) 관련 문서를 시스템 프롬프트에 포함
# 2) "문서에 없으면 확인 불가"로 환각 차단
# 3) 이 과정을 자동화한 것이 RAG 파이프라인
sys_rag = (
    "아래 제공된 문서 내용만을 근거로 답하세요. "
    "문서에 없는 내용은 '문서에서 확인 불가'라고 답하세요.\n\n"
    f"[참고 문서]\n{REAL_CONTEXT}"
)

ans_rag, _ = chat(
    system_prompt=sys_rag,
    user_prompt="2024년 연구비 집행 지침에서 바뀐 주요 내용은 무엇인가요?",
    temperature=RAG_TEMPERATURE,
)
print(f"\n[RAG 방식] 문서 컨텍스트 주입 후 응답:")
print(ans_rag)
print("\n-> 차이: 문서를 프롬프트에 주입 -> 근거 있는 정확한 답변")
print("   이 '문서 주입' 과정을 자동화하는 파이프라인 = RAG")
print("   오후 실습: PDF 로드 -> 청킹 -> 벡터 검색 -> 자동 주입")


# ── 블록 4: API 단독 vs RAG + API 적합도 비교 시각화 ──────────
section("블록 4. API 단독 vs RAG + API 연구 업무 적합도 비교")

categories = list(API_SCORES.keys())
api_vals   = [API_SCORES[c] for c in categories]
rag_vals   = [RAG_SCORES[c] for c in categories]

x = np.arange(len(categories))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, api_vals, w, label="API 단독",  color="#B5D4F4", edgecolor="none")
b2 = ax.bar(x + w/2, rag_vals, w, label="RAG + API", color="#9FE1CB", edgecolor="none")

for bar, v in zip(b1, api_vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.08,
            str(v), ha='center', fontsize=10, color="#185FA5")
for bar, v in zip(b2, rag_vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.08,
            str(v), ha='center', fontsize=10, color="#0F6E56")

ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylabel("적합도 (1~5)")
ax.set_ylim(0, 6.5)
ax.set_title("API 단독 vs RAG + API\n연구 업무 적합도 비교")
ax.legend()

# 가장 차이가 큰 항목 강조 표시
diffs = [r - a for a, r in zip(api_vals, rag_vals)]
max_diff_idx = np.argmax(diffs)
ax.annotate(
    f"차이 최대\n(+{diffs[max_diff_idx]})",
    xy=(max_diff_idx + w/2, rag_vals[max_diff_idx]),
    xytext=(max_diff_idx + w/2 + 0.5, rag_vals[max_diff_idx] + 0.5),
    arrowprops=dict(arrowstyle="->", color="#D85A30"),
    fontsize=9, color="#D85A30",
)

plt.tight_layout()
plt.savefig("api_vs_rag.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n핵심 요약:")
for cat, a, r in zip(categories, api_vals, rag_vals):
    diff = r - a
    bar  = "+" * diff if diff > 0 else ""
    print(f"  {cat:10s}: API={a}  RAG+API={r}  {bar}")

print("\n-> 즉시 사용성은 API 단독이 높으나")
print("   연구 업무 핵심(내부 문서, 근거 제시)에서 RAG+API가 압도적 우위")
print("-> 오후 실습: 규정 PDF 3종으로 RAG 파이프라인을 직접 구현합니다")